In [2]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]


In [3]:
import cryptography

print("Cryptography version:")
print(cryptography.__version__)

Cryptography version:
50.0.1


In [4]:
from cryptography.hazmat.primitives.asymmetric import rsa

private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

print("Private key created")
print("Public key created")

Private key created
Public key created


In [5]:
message = b"This is my first PKI digital signature."

print(message)

b'This is my first PKI digital signature.'


In [6]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding

signature = private_key.sign(
    message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print("Message signed successfully!")
print("Signature length:", len(signature), "bytes")

Message signed successfully!
Signature length: 256 bytes


In [7]:
public_key.verify(
    signature,
    message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print("Signature is VALID!")

Signature is VALID!


In [8]:
tampered_message = b"This is my first PKI digital signature. HACKED!"

print(tampered_message)

b'This is my first PKI digital signature. HACKED!'


In [9]:
public_key.verify(
    signature,
    tampered_message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

InvalidSignature: 

In [10]:
InvalidSignature


NameError: name 'InvalidSignature' is not defined

In [11]:
"This is my first PKI digital signature."

'This is my first PKI digital signature.'

In [12]:
"This is my first PKI digital signature. HACKED!"

'This is my first PKI digital signature. HACKED!'

In [13]:
from cryptography.exceptions import InvalidSignature

try:
    public_key.verify(
        signature,
        tampered_message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    print("Signature is VALID!")

except InvalidSignature:
    print("WARNING: Signature is INVALID!")
    print("The message may have been modified.")

The message may have been modified.


In [16]:
       
try:
    public_key.verify(
        signature,
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    print("✅ SIGNATURE VALID")
    print("✅ Message has not been modified")

except InvalidSignature:
    print("❌ SIGNATURE INVALID")

✅ SIGNATURE VALID
✅ Message has not been modified


In [17]:
from cryptography.hazmat.primitives import hashes


In [18]:
message = b"Hello PKI"

digest = hashes.Hash(hashes.SHA256())
digest.update(message)

hash_value = digest.finalize()

print("SHA-256 hash:")
print(hash_value.hex())

SHA-256 hash:
701b36d9354cecf3ecfaf714a4a5065c7a4e4c96e68882e9ef7b66e8251217c6


In [19]:
message2 = b"Hello PKI!"

digest2 = hashes.Hash(hashes.SHA256())
digest2.update(message2)

hash_value2 = digest2.finalize()

print("Original:")
print(hash_value.hex())

print("\nModified:")
print(hash_value2.hex())

Original:
701b36d9354cecf3ecfaf714a4a5065c7a4e4c96e68882e9ef7b66e8251217c6

Modified:
51bea2c9eab4dbcfcc0e090b8a2d71dce92bffbda8f5c00253f520bc9ae3c527


In [20]:
print("Are the hashes identical?", hash_value == hash_value2)

Are the hashes identical? False


In [21]:
message = b"This is my first PKI digital signature."

digest = hashes.Hash(hashes.SHA256())
digest.update(message)

message_hash = digest.finalize()

print("Message:")
print(message.decode())

print("\nSHA-256:")
print(message_hash.hex())

Message:
This is my first PKI digital signature.

SHA-256:
71286d585e9e2699ab152c34de8c01995b7a4660c19ab2be59ff238118d47172


In [22]:
original = b"certificate=www.example.com"
modified = b"certificate=www.attacker.com"

def calculate_sha256(data):
    digest = hashes.Hash(hashes.SHA256())
    digest.update(data)
    return digest.finalize().hex()

print("Original hash:")
print(calculate_sha256(original))

print("\nModified hash:")
print(calculate_sha256(modified))

Original hash:
e842fa471c62b115d045b5ae62403fbb11c76ec0d4e40fe5df3e879324187af8

Modified hash:
e7bd60f2d83e6ceea26069bd1248cc286e56289a930cb77ee8793e700024624e


In [23]:
from cryptography import x509
from cryptography.x509.oid import NameOID

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa

from datetime import datetime, timedelta, timezone

In [24]:
from cryptography import x509
from cryptography.x509.oid import NameOID

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa

from datetime import datetime, timedelta, timezone

In [25]:
root_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

print("Root CA private key generated.")

Root CA private key generated.


In [26]:
root_public_key = root_private_key.public_key()

print("Root CA public key generated.")

Root CA public key generated.


In [27]:
subject = issuer = x509.Name([
    x509.NameAttribute(NameOID.COUNTRY_NAME, "IN"),
    x509.NameAttribute(
        NameOID.ORGANIZATION_NAME,
        "My PKI Lab"
    ),
    x509.NameAttribute(
        NameOID.COMMON_NAME,
        "My Root CA"
    ),
])

print(subject)

<Name(CN=My Root CA,O=My PKI Lab,C=IN)>


In [28]:
certificate = (
    x509.CertificateBuilder()
    .subject_name(subject)
    .issuer_name(issuer)
    .public_key(root_public_key)
    .serial_number(x509.random_serial_number())
    .not_valid_before(
        datetime.now(timezone.utc)
    )
    .not_valid_after(
        datetime.now(timezone.utc) + timedelta(days=3650)
    )
    .add_extension(
        x509.BasicConstraints(
            ca=True,
            path_length=None
        ),
        critical=True
    )
    .sign(
        private_key=root_private_key,
        algorithm=hashes.SHA256()
    )
)

print("X.509 Root CA certificate created!")

X.509 Root CA certificate created!


In [29]:
.subject_name(subject)

SyntaxError: invalid syntax (4179917631.py, line 1)

In [30]:
certificate = (
    x509.CertificateBuilder()
    .subject_name(subject)
    .issuer_name(issuer)
    .public_key(root_public_key)
    .serial_number(x509.random_serial_number())
    .not_valid_before(
        datetime.now(timezone.utc)
    )
    .not_valid_after(
        datetime.now(timezone.utc) + timedelta(days=3650)
    )
    .add_extension(
        x509.BasicConstraints(
            ca=True,
            path_length=None
        ),
        critical=True
    )
    .sign(
        private_key=root_private_key,
        algorithm=hashes.SHA256()
    )
)

print("X.509 Root CA certificate created!")

X.509 Root CA certificate created!


In [31]:
print("Subject:")
print(certificate.subject)

print("\nIssuer:")
print(certificate.issuer)

print("\nSerial Number:")
print(certificate.serial_number)

print("\nValid From:")
print(certificate.not_valid_before_utc)

print("\nValid Until:")
print(certificate.not_valid_after_utc)

Subject:
<Name(CN=My Root CA,O=My PKI Lab,C=IN)>

Issuer:
<Name(CN=My Root CA,O=My PKI Lab,C=IN)>

Serial Number:
636984385843550350591936023735001720874230824031

Valid From:
2026-09-06 13:39:05+00:00

Valid Until:
2036-09-03 13:39:05+00:00
